# Hybrid search: BM25 + dense + fusion
BM25 from scratch, an LSA dense retriever, reciprocal rank fusion and weighted fusion on keyword vs paraphrased queries.

## 1. Corpus and labeled queries

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from data import DOCS, BACKGROUND, QUERIES
from text import tokenize
print(len(DOCS), 'docs |', len(BACKGROUND), 'background lines |', len(QUERIES), 'queries')
for q in QUERIES[:2] + QUERIES[12:14]:
    print(q)
print(tokenize(DOCS['d02']))

## 2. BM25 from scratch
score = sum over query terms of idf * tf*(k1+1) / (tf + k1*(1 - b + b*len/avglen))

In [ ]:
from bm25 import BM25
bm = BM25(DOCS)
for q in ['ERR-4012', 'how do I get my money back after cancelling']:
    s = bm.scores(q); top = np.argsort(-s)[:3]
    print(q, '->', [(bm.ids[i], round(float(s[i]), 2)) for i in top])

## 3. Dense retriever (TF-IDF -> SVD) fitted on background text only
Codes like `ERR-4012` are out of vocabulary for the dense model, just as rare strings are for real encoders.

In [ ]:
from dense import LSADense
de = LSADense(DOCS, BACKGROUND, dim=24)
print('vocab', len(de.vocab), '| explained var %.3f' % de.explained, '| doc-token OOV rate %.2f' % de.doc_oov_rate)
for q in ['ERR-4012', 'how do I get my money back after cancelling']:
    print(q, '->', de.search(q, 3), '| query vector norm', round(float(np.linalg.norm(de.embed(q))), 3))

## 4. Fusion: RRF and weighted

In [ ]:
from fusion import rrf, weighted
q = 'screen keeps blinking'
print('bm25    ', bm.search(q, 5))
print('dense   ', de.search(q, 5))
print('rrf     ', rrf([bm.search(q, 10), de.search(q, 10)])[:5])
print('weighted', weighted(bm.ids, bm.scores(q), de.scores(q), 0.5)[:5])

## 5. Evaluate recall@k / MRR / nDCG by query type

In [ ]:
from retrieval import build, methods, run
bm, de = build()
for name, fn in methods(bm, de).items():
    agg = run(fn)['aggregate']
    print(f"{name:14s}", {g: agg[g]['ndcg@5'] for g in ('keyword', 'paraphrase', 'all')}, 'MRR', agg['all']['mrr@10'])

## 6. Full smoke run
`python run_smoke.py` rewrites `results/` (RESULTS.md, metrics.json, JSON.shot, SVG plots).